# Step 5: 端到端闭环 mini 项目 + 前瞻（finale）

**目标**：把整条 M2→M3→M4 链路串成端到端闭环——理解整条链路的**唯一交接物**（compressed-tensors 产物目录）、能按场景约束（显存/算力/精度/硬件）决策选方法（M1.6 三范式 + M3.8 四维 Pareto）、组装可复现**交付物四件套**（recipe + uv.lock + serve 命令 + 压测报告），并前瞻 QAT/NVFP4。

**对应 OUTLINE 课时**：4.6 端到端闭环 mini 项目（~90min）+ 4.7 QAT/NVFP4 前瞻（~25min，finale）。

> **闭环认知**：整条链路之所以能解耦（量化/调优/部署各模块独立），全靠那个**唯一的交接物**——compressed-tensors 产物目录（`config.json` 的 `quantization_config` + 量化权重）。M2 产出它、M3 调优它、M4 读它部署，全程不传 Python 对象、只传一个目录。


## 学完应能讲清（学完本节应能口头回答）

1. 整条链路的**唯一交接物**是什么？（compressed-tensors 产物目录：`config.json` 的 `quantization_config` + 量化权重）为什么这个交接物让量化/调优/部署解耦？（提示：各模块只读写一个目录，不传 Python 对象/不共享内存状态；M2 产、M3 调、M4 读）
2. **SmoothQuant 端到端部署闭环**怎么串？（s1 加载 W8A8 产物 auto 识别 → s2 多卡 serve → s3 FP16 vs W8A8 压测 → s5 解读压测结果生成部署总结）每一步的交付物是什么？
3. 怎么**解读 s3 压测结果**判断 SmoothQuant W8A8 部署是否达标？（提示：对比 FP16 基线的吞吐/显存/TTFT——显存省了 X GB、吞吐提/降了 Y%、TTFT 变化在可接受范围即达标；这是 `build_deploy_summary` 的核心）
4. 可复现**交付物四件套**是哪四件（recipe + 各模块 uv.lock + serve 命令 + 压测报告）？为什么固定 seed/prompt/batch？（提示：四件套锁定量化逻辑+环境版本+部署参数+性能数字；不固定则复现结果漂移，对比无意义）
5. QAT 为什么本课不动手（万亿 token 重训成本，仅蒸馏/继续训练）？NVFP4 为什么 H200 跑不了（Blackwell 第五代 Tensor Core 独有）？

In [ ]:
%%capture
import pathlib, os, json
import ipytest
ipytest.autoconfig()


In [ ]:
# Setup cell（cwd 无关路径解析）。M4 跨模块读 M2/M3 7B 量化产物做闭环串接。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT   = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT       = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_COURSE = MODULE_ROOT.parent
M2_OUT = REPO_COURSE / "m2-quant-pipeline" / "out"      # 唯一交接物：量化产物目录
M3_OUT = REPO_COURSE / "m3-tuning-eval" / "out"
print("MODULE_ROOT =", MODULE_ROOT)
print("M2_OUT =", M2_OUT, "| exists:", M2_OUT.exists())


## 原理：唯一交接物 + SmoothQuant 端到端部署闭环 + 交付物四件套

**整条链路的唯一交接物**：compressed-tensors **产物目录**（`config.json` 的 `quantization_config` + 量化权重 `.safetensors`）。这是 M2→M3→M4 解耦的关键——各模块只读写这一个目录，不传 Python 对象、不共享内存状态：

```
M2 量化     ──产出──▶  out/qwen7b-smoothquant/   (config + 权重，SmoothQuant W8A8)
                              │ 唯一交接物（目录）
M3 调优     ──读改──▶  out/s5_tuned, s6_final    (layer fallback / smoothing_strength 调优，同 W8A8 格式)
                              │ 同一目录格式
M4 部署     ──读───▶  vllm serve <dir>           (vLLM 读 config.json 自动适配)
```

为什么这让链路解耦：M2 用 quant env（llmcompressor）、M4 用 vllm env，两个 env 的 transformers 版本都不同——但它们通过一个**磁盘上的目录**交接，互不干扰。本模块全程只交接 SmoothQuant W8A8 这一个产物。

**SmoothQuant 端到端部署闭环（M4 四步，本节 finale 把它们串起来）**：

| Step | 动作 | 交付物 |
|---|---|---|
| s1 | 加载 W8A8 产物，验 vLLM auto 识别 | config 结构 + kernel 速查 |
| s2 | 多卡 `vllm serve`（纯 TP） | serve 命令模板 + TP 选型 |
| s3 | FP16 vs W8A8 压测（两向） | 吞吐/TTFT/显存对比（`out/s3_bench_compare.json`）|
| **s5** | **解读压测结果 + 组装交付物** | **部署总结 + 可复现四件套** |

`build_deploy_summary` 就是 finale 的核心——读 s3 的压测结果，生成"SmoothQuant vs FP16 部署对比 + 是否达标"的结构化总结，让闭环的"值不值得上线 W8A8"这个判断**可量化、可复现**。

**交付物四件套（可复现）**：
1. **recipe**：量化 recipe（SmoothQuant 两段式：`SmoothQuantModifier(smoothing_strength=α) + GPTQModifier(targets='Linear', scheme='W8A8', ignore=...)`）——锁定量化逻辑。
2. **uv.lock**（各模块）：锁定环境版本（vllm/llmcompressor/compressed-tensors/transformers）——复现的关键（跨平台精确版本图）。
3. **serve 命令**：部署参数（`vllm serve <dir> --tensor-parallel-size 2 ...`）——锁定部署配置。
4. **压测报告**：性能数字（FP16 vs W8A8 吞吐/TTFT/显存对比）+ 部署总结——验证交付有效。

**为什么固定 seed/prompt/batch**：量化/调优/评测/压测每个环节都有随机性（校准采样、生成采样、压测请求分布）。不固定则每次复现结果漂移，"量化前后对比"无意义。固定 seed(42) + prompt 集 + batch/num_prompts 才能拿到可重复的对比。

> **不做多算法选型**：本课已选定 SmoothQuant 作为端到端主线（M1/M2 讲清了三种方法的选型判断），M4 不重复"给我场景选方法"的决策树——直接走 SmoothQuant 部署闭环，把精力放在"如何把 W8A8 调优好、部署好、压测好、交付好"。

## 亲手摸一摸：整条链路产物串接（M2/M3/M4）

看 M2 `out/` 的产物目录如何串到 M4——config.json + 量化权重就是那个唯一交接物。


In [ ]:
# 摸一摸：M2/M3 的 SmoothQuant W8A8 产物目录结构（唯一交接物）
products = {
    "M2 SmoothQuant (全量化)": M2_OUT / "qwen7b-smoothquant",
    "M3 final (调优后)":        M3_OUT / "s6_final",
}
print("=== 整条链路的唯一交接物：SmoothQuant W8A8 compressed-tensors 产物目录 ===")
for name, d in products.items():
    if not d.exists():
        print("[%s] %s 不存在（先跑 M2 s5 / M3 s6）" % (name, d))
        continue
    files = sorted(p.name for p in d.iterdir() if not p.name.startswith("."))
    has_config = (d / "config.json").exists()
    has_weights = any(f.endswith(".safetensors") for f in files)
    print("\n[%s] %s" % (name, d))
    print("  文件:", files[:6])
    print("  config.json 含 quantization_config:", has_config, "| 量化权重 .safetensors:", has_weights)
    print("  -> M4 只需 vllm serve %s 即可部署（读 config.json 自动适配）" % d)
print("\n=== 链路串接（SmoothQuant W8A8 一条主线）===")
print("  M2 量化   -> out/qwen7b-smoothquant/  (产 W8A8 config + 权重)")
print("  M3 调优   -> out/s5_tuned, s6_final    (layer fallback / α 调优，同 W8A8 格式)")
print("  M4 部署   -> vllm serve <dir>          (读 config.json auto 适配)")
print("  全程只交接「一个 W8A8 目录」，三模块解耦。")

## 本步填空（2 个）

1. **`build_deploy_summary(bench_result)`**（业务型，**新增**）— 读 s3 的压测结果（FP16 vs SmoothQuant W8A8），生成结构化部署总结：显存省了多少、吞吐提/降了多少、TTFT 变化、**是否达标**（达标的判据由你定，docstring 给方向）。**为什么这么设计（填前先想）**：finale 的核心交付——把 s3 的裸压测数字"翻译"成"值不值得上线 W8A8"的可量化判断，补足端到端闭环的最后一环。这是 s3 压测的下游消费者，不是查表。
2. **`build_delivery_bundle(recipe, uv_locks, serve_cmd, bench_report)`**（组装型）— 组装可复现交付物四件套 manifest（dict）。**为什么这么设计**：组装型——把零散交付物结构化，体现"可复现"的工程约定；缺件报错。

> **已删 `build_decision_tree`**：原 finale 有个"场景约束→选方法（FP8/AWQ/SmoothQuant）"的决策树填空，与"本模块只用 SmoothQuant 一条主线"直接冲突（M1/M2 已讲过选型），故删除。改用 `build_deploy_summary` 补足"解读压测结果→部署总结"的端到端交付。

In [ ]:
def build_deploy_summary(bench_result):
    """读 s3 压测结果（FP16 vs SmoothQuant W8A8），生成结构化部署总结（dict）。

    参数 bench_result：s3 产出的压测结果 dict，形如：
        {
          "FP16":         {"throughput_tps": 1000.0, "ttft_ms": 40.0, "weight_mem_gb": 14.0},
          "SmoothQuant":  {"throughput_tps": 1300.0, "ttft_ms": 48.0, "weight_mem_gb": 7.5},
        }
      （字段名容错：throughput_tps / throughput / tps 任一表示吞吐；ttft_ms / ttft 任一表示 TTFT；
        weight_mem_gb / weight_gb / mem_gb 任一表示权重显存。缺某方法或缺字段要优雅降级，不崩。）

    为什么这么设计（填前先想）：
    - 业务型——finale 核心交付。s3 产出的是裸数字，你要把它"翻译"成可量化的部署判断：
        显存省了多少 GB（FP16.weight_mem - W8A8.weight_mem）
        吞吐比（W8A8.tps / FP16.tps）—— >1 是提速、<1 是拖慢
        TTFT 变化（W8A8.ttft - FP16.ttft，单位 ms）
        **是否达标**：达标判据由你定（docstring 只给方向，不给逐字阈值）——常见做法是
          「显存省了（weight_mem 减少）且吞吐不显著下降（如吞吐比 >= 0.9）且 TTFT 不暴涨（如增幅 < 50%）」
          三者都满足 -> meets_bar=True。
    - 缺 FP16 或 SmoothQuant 任一 -> meets_bar=False、reason 注明"缺基线/缺量化数据，无法对比"。

    返回：dict，至少含：
        method="SmoothQuant W8A8", baseline="FP16",
        weight_mem_saved_gb, throughput_ratio, ttft_delta_ms, meets_bar(bool), reason(str)
    """
    # TODO:
    #   1) 从 bench_result 取 FP16 / SmoothQuant 两个 dict（大小写/别名容错）。
    #   2) 取各字段（throughput/ttft/weight_mem 的别名容错）。
    #   3) 算 weight_mem_saved_gb / throughput_ratio / ttft_delta_ms。
    #   4) 按你定的判据算 meets_bar + reason。
    #   5) 返回 dict（上述键）。缺数据 -> meets_bar=False + reason 注明缺什么。
    raise NotImplementedError

In [ ]:
%%ipytest -qq
# L1 测试（build_deploy_summary）——填完 build_deploy_summary 立即单独跑此 cell 验证（不依赖 build_delivery_bundle）。

def test_summary_w8a8_beats_baseline():
    bench = {
        "FP16":        {"throughput_tps": 1000.0, "ttft_ms": 40.0, "weight_mem_gb": 14.0},
        "SmoothQuant": {"throughput_tps": 1300.0, "ttft_ms": 48.0, "weight_mem_gb": 7.5},
    }
    s = build_deploy_summary(bench)
    assert s["method"] == "SmoothQuant W8A8" and s["baseline"] == "FP16"
    assert abs(s["weight_mem_saved_gb"] - 6.5) < 1e-9          # 14 - 7.5
    assert abs(s["throughput_ratio"] - 1.3) < 1e-9             # 1300/1000
    assert abs(s["ttft_delta_ms"] - 8.0) < 1e-9                # 48 - 40
    assert s["meets_bar"] is True                              # 省显存 + 提速 + TTFT 增幅小

def test_summary_w8a8_slowdown_not_meet_bar():
    # W8A8 吞吐显著下降 -> 不达标
    bench = {
        "FP16":        {"throughput_tps": 1000.0, "ttft_ms": 40.0, "weight_mem_gb": 14.0},
        "SmoothQuant": {"throughput_tps": 500.0,  "ttft_ms": 90.0, "weight_mem_gb": 7.5},
    }
    s = build_deploy_summary(bench)
    assert s["meets_bar"] is False
    assert s["throughput_ratio"] < 1.0

def test_summary_field_aliases():
    # 字段别名容错：throughput/ttft/weight
    bench = {
        "FP16":        {"tps": 1000.0, "ttft": 40.0, "weight_gb": 14.0},
        "SmoothQuant": {"throughput": 1100.0, "ttft_ms": 45.0, "mem_gb": 7.5},
    }
    s = build_deploy_summary(bench)
    assert s["weight_mem_saved_gb"] > 0
    assert s["throughput_ratio"] > 1.0

def test_summary_missing_baseline():
    # 缺 FP16 基线 -> 无法对比，meets_bar=False
    s = build_deploy_summary({"SmoothQuant": {"throughput_tps": 1300.0, "ttft_ms": 48.0, "weight_mem_gb": 7.5}})
    assert s["meets_bar"] is False
    assert "FP16" in s["reason"] or "基线" in s["reason"]

def test_summary_missing_quant():
    # 缺 SmoothQuant 量化数据 -> 无法对比
    s = build_deploy_summary({"FP16": {"throughput_tps": 1000.0, "ttft_ms": 40.0, "weight_mem_gb": 14.0}})
    assert s["meets_bar"] is False

In [ ]:
def build_delivery_bundle(recipe, uv_locks, serve_cmd, bench_report):
    """组装可复现交付物四件套 manifest（dict）。

    四件套参数：
    - recipe：量化 recipe（dict 或路径 str）—— 锁定量化逻辑
    - uv_locks：各模块 uv.lock（dict module->path）—— 锁定环境版本
    - serve_cmd：部署命令（str）—— 锁定部署配置
    - bench_report：压测报告（dict 或路径 str）—— 验证交付有效

    为什么这么设计（填前先想）：
    - 组装型——把零散交付物结构化成一份 manifest，体现"可复现"的工程约定。
    - 缺任一件（None 或空）-> ValueError，明确点出缺哪个（可复现不能有缺口）。
    - 加 bundle_version + repro_note（固定 seed/prompt/batch 说明）。

    返回：dict（含 recipe/uv_locks/serve_cmd/bench_report + bundle_version + repro_note）。
    """
    # TODO:
    #   1) 检查四件套是否齐全：None 或空（str/list/dict/tuple len 0）-> ValueError('交付物缺失：...')。
    #   2) 组 dict 返回（四件套 + 'bundle_version': '1.0' + repro_note）。
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（build_delivery_bundle）——填完 build_delivery_bundle 立即单独跑此 cell 验证（不依赖 build_deploy_summary）。

def test_delivery_bundle_complete():
    bundle = build_delivery_bundle(
        recipe={"scheme": "W8A8", "smoothing_strength": 0.8, "targets": "Linear"},
        uv_locks={"m2": "course/m2-quant-pipeline/uv.lock",
                  "m3": "course/m3-tuning-eval/uv.lock",
                  "m4": "course/m4-deploy-loop/uv.lock"},
        serve_cmd="vllm serve /models/qwen7b-smoothquant --tensor-parallel-size 2",
        bench_report={"throughput_tps": 1300.0, "ttft_ms": 48.0, "weight_mem_gb": 7.5},
    )
    assert bundle["recipe"]["scheme"] == "W8A8"
    assert "m4" in bundle["uv_locks"]
    assert "vllm serve" in bundle["serve_cmd"]
    assert bundle["bench_report"]["throughput_tps"] == 1300.0
    assert bundle["bundle_version"] == "1.0"
    assert "seed" in bundle["repro_note"]

def test_delivery_bundle_missing_recipe():
    import pytest
    with pytest.raises(ValueError, match="recipe"):
        build_delivery_bundle(recipe=None,
            uv_locks={"m4": "x"}, serve_cmd="vllm serve x", bench_report={"t": 1})

def test_delivery_bundle_missing_serve_cmd():
    import pytest
    with pytest.raises(ValueError, match="serve_cmd"):
        build_delivery_bundle(recipe={"s": 1}, uv_locks={"m4": "x"}, serve_cmd="", bench_report={"t": 1})

def test_delivery_bundle_missing_bench_report():
    import pytest
    with pytest.raises(ValueError, match="bench_report"):
        build_delivery_bundle(recipe={"s": 1}, uv_locks={"m4": "x"},
            serve_cmd="vllm serve x", bench_report=None)

def test_delivery_bundle_missing_uv_locks():
    import pytest
    with pytest.raises(ValueError, match="uv_locks"):
        build_delivery_bundle(recipe={"s": 1}, uv_locks={},
            serve_cmd="vllm serve x", bench_report={"t": 1})

## L2（CPU）：解读压测结果（部署总结）+ 交付物组装（纯逻辑）

L2 验 `build_deploy_summary` 解读 s3 压测结果（FP16 vs SmoothQuant W8A8 → 是否达标）+ `build_delivery_bundle` 组装四件套（CPU 可跑，纯逻辑）。

In [ ]:
## L2：解读压测结果（部署总结）+ 组装交付物四件套（纯逻辑）
print("=== L2：解读 FP16 vs SmoothQuant W8A8 压测结果 -> 部署总结 ===")
# 示例压测结果（真人跑 L3 后用 out/s3_bench_compare.json 的真实数字替换）
bench_demo = {
    "FP16":        {"throughput_tps": 1000.0, "ttft_ms": 40.0, "weight_mem_gb": 14.0},
    "SmoothQuant": {"throughput_tps": 1300.0, "ttft_ms": 48.0, "weight_mem_gb": 7.5},
}
summary = build_deploy_summary(bench_demo)
print("  显存省了: %.1f GB" % summary["weight_mem_saved_gb"])
print("  吞吐比 (W8A8/FP16): %.2f%s" % (summary["throughput_ratio"],
      "（提速）" if summary["throughput_ratio"] >= 1.0 else "（拖慢）"))
print("  TTFT 变化: %+.1f ms" % summary["ttft_delta_ms"])
print("  是否达标: %s（%s）" % (summary["meets_bar"], summary["reason"]))

print("\n=== L2：组装 SmoothQuant 部署交付物四件套 ===")
bundle = build_delivery_bundle(
    recipe={"scheme": "W8A8", "smoothing_strength": 0.8, "targets": "Linear", "ignore": ["lm_head"]},
    uv_locks={"m2": "course/m2-quant-pipeline/uv.lock",
              "m3": "course/m3-tuning-eval/uv.lock",
              "m4": "course/m4-deploy-loop/uv.lock"},
    serve_cmd="vllm serve /models/qwen7b-smoothquant --tensor-parallel-size 2 --enable-prefix-caching",
    bench_report=summary,
)
print("  bundle_version:", bundle["bundle_version"])
print("  recipe:", bundle["recipe"])
print("  uv_locks 模块:", list(bundle["uv_locks"].keys()))
print("  repro_note:", bundle["repro_note"])
json.dump({"summary": summary, "bundle": bundle},
          open(OUT_ROOT / "s5_deploy_summary.json", "w"), indent=2, ensure_ascii=False)
# 验：W8A8 典型情况达标
assert summary["meets_bar"] is True, "典型 W8A8（提速+省显存+TTFT 小增）应达标"
assert bundle["bundle_version"] == "1.0" and "seed" in bundle["repro_note"]
print("\nL2 通过：压测解读 + 四件套组装正确（部署总结存 out/s5_deploy_summary.json）。")

## L3（可选）：链路串接演示（s3 压测结果 → 部署总结 → 交付物）

L3 可选：读 s3 真压测结果 → `build_deploy_summary` 解读 → `build_delivery_bundle` 组装，演示 SmoothQuant 端到端部署闭环的最后一步（s3 真压测数字换成部署总结）。纯逻辑已在 L1+L2 验，L3 留作真人串接演示（缺 s3 产物时用示例数据演示流程）。

> **L3 双守卫**：`torch.cuda.is_available() and not os.environ.get('SKIP_L3')`——reviewer 执行验证设 `SKIP_L3=1` 跳过；真人跑时不设，L3 实证。

In [ ]:
import torch, os, json

def run_l3_e2e_link():
    # 演示：读 s3 真压测结果 -> build_deploy_summary -> build_delivery_bundle（不真起服务，只串逻辑）
    bench_path = OUT_ROOT / "s3_bench_compare.json"
    if bench_path.exists():
        bench = json.loads(bench_path.read_text())
        # s3 存的是 {FP16: {...}, SmoothQuant: {...}}；字段名规整为 build_deploy_summary 期望的
        print("[L3] 读 s3 真压测结果 %s" % bench_path)
    else:
        print("[L3] s3 压测结果缺失（out/s3_bench_compare.json），用示例数据演示闭环逻辑。")
        bench = {
            "FP16":        {"throughput_tps": 1000.0, "ttft_ms": 40.0, "weight_mem_gb": 14.0},
            "SmoothQuant": {"throughput_tps": 1300.0, "ttft_ms": 48.0, "weight_mem_gb": 7.5},
        }
    summary = build_deploy_summary(bench)
    print("[L3] SmoothQuant W8A8 部署总结：达标=%s" % summary["meets_bar"])
    print("     显存省 %.1f GB | 吞吐比 %.2f | TTFT %+d ms" % (
        summary["weight_mem_saved_gb"], summary["throughput_ratio"], summary["ttft_delta_ms"]))
    # 用 M3 final 产物作为部署对象（调优后的 W8A8）
    prod = M3_OUT / "s6_final"
    if not (prod / "config.json").exists():
        prod = M2_OUT / "qwen7b-smoothquant"
    serve_cmd = "vllm serve %s --tensor-parallel-size 2 --enable-prefix-caching" % prod
    bundle = build_delivery_bundle(
        recipe={"scheme": "W8A8", "source": str(prod)},
        uv_locks={"m4": "course/m4-deploy-loop/uv.lock"},
        serve_cmd=serve_cmd,
        bench_report=summary,
    )
    print("[L3] 端到端闭环交付物 bundle_version=%s | repro=%s" % (
        bundle["bundle_version"], bundle["repro_note"][:50]))

if torch.cuda.is_available() and not os.environ.get('SKIP_L3'):
    run_l3_e2e_link()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（链路逻辑 L1+L2 已验；真人跑时不设 SKIP_L3 可串接真压测结果）。")

## 前瞻：QAT 与 NVFP4（纯概念，不填空）

整条 SmoothQuant 端到端部署闭环到此走通。最后前瞻两个"本课不动手但要知道方向"的技术：

### QAT（Quantization-Aware Training）
- **思想**：训练时插入 fake-quant（前向模拟量化截断）+ **直通估计器（STE，Straight-Through Estimator）**（反向把量化不可导的梯度"直通"过去），让模型在训练中就适应量化误差——理论精度上限高于 PTQ。
- **为什么本课不动手**：LLM 的 QAT 需要在**接近原始训练规模的数据（万亿 token）**上重新训练（或继续训练），成本对工业级 7B+ 模型不可接受。QAT 仅在**蒸馏 / 继续训练**场景（已有训练 pipeline、只做量化适配微调）才划算。本课定位 PTQ 工业落地，故 QAT 仅作概念。
- **方向**：未来若开源大模型 QAT 后的检查点普及（如社区产 QAT 版 Qwen/Llama），可直接拿来部署——但那是消费别人的 QAT 产物，不是自己跑 QAT。

### NVFP4 / FP4
- **是什么**：4-bit 浮点（NVFP4 微缩浮点格式），**Blackwell（B100/B200）第五代 Tensor Core 独有**。
- **为什么 H200 跑不了**：Hopper（H200/H100）只有 FP8 Tensor Core，**没有 FP4 Tensor Core**——零 FP4 吞吐。NVFP4 是 Blackwell 世代（2025+）的硬件红利，H200 上跑 FP4 只能软件模拟（慢）。
- **方向**：当 Blackwell 数据中心卡普及，FP4（比 FP8 再省一半显存 + 更高吞吐）会成为新首选——本课的 SmoothQuant W8A8 部署流水线（同样的 compressed-tensors 产物格式 + vLLM 声明式部署）可平滑迁移，只是 scheme/kernel 换掉。

> **产物可发布到 HF Hub**：本课产出的 compressed-tensors 产物目录可用 `huggingface_hub` 的 `upload_folder` 发布到 Hub 供他人复现/部署。本课不实操（Hub 发布偏运维/分享动作、非核心部署能力）。

---

**SmoothQuant 端到端部署闭环达成**：从 M1 激活离群点原理 → M2 SmoothQuant 量化流水线 → M3 精度调优（layer fallback + smoothing_strength）+ 评测 → **M4 vLLM 声明式部署 + 压测 + 排错 + 部署总结**，你已具备在 H200×8 上把一个 7B 模型用 **SmoothQuant W8A8** 量化、调优、部署、压测、排错、并给出可量化部署判断（`build_deploy_summary`）的完整工业能力。**唯一交接物**（compressed-tensors W8A8 产物目录）+ **交付物四件套**（recipe + uv.lock + serve + bench/summary）= 可复现的端到端量化部署流水线。